In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

# =========================
# 1) Load dataset
# =========================
df = pd.read_csv("processed_ecommerce_data.csv")

TARGET = "Churn"
ID_COL = "CustomerID"  # drop this from features

X = df.drop(columns=[TARGET])
if ID_COL in X.columns:
    X = X.drop(columns=[ID_COL])
y = df[TARGET].astype(int)

# =========================
# 2) Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# 3) Preprocess (same style as your notebook)
# =========================
num_cols = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="drop"
)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

# =========================
# 4) SMOTE on train only
# =========================
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_prep, y_train)

# =========================
# 5) Helpers for Keras
# =========================
def to_dense(x):
    return x.toarray() if hasattr(x, "toarray") else np.asarray(x)

Xtr = to_dense(X_train_res).astype("float32")
ytr = np.asarray(y_train_res).astype("float32")
Xte = to_dense(X_test_prep).astype("float32")
yte = np.asarray(y_test).astype("float32")

input_dim = Xtr.shape[1]

early = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=5,
    restore_best_weights=True
)

def evaluate_nn(name, model, X_eval, y_eval):
    y_proba = model.predict(X_eval).ravel()
    y_pred = (y_proba >= 0.5).astype(int)

    print(f"\n=== {name} ===")
    print(classification_report(y_eval.astype(int), y_pred))
    print("ROC AUC:", roc_auc_score(y_eval, y_proba))

c:\Users\Adithya\Desktop\public\.venv\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


## MLP (best-fit NN for tabular data without time series analysis)

In [4]:
mlp = tf.keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])

mlp.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(name="auc")]
)

mlp.fit(
    Xtr, ytr,
    validation_split=0.2,
    epochs=50,
    batch_size=256,
    callbacks=[early],
    verbose=1
)

evaluate_nn("Neural Net (MLP)", mlp, Xte, yte)


Epoch 1/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - auc: 0.7722 - loss: 0.6134 - val_auc: 0.0000e+00 - val_loss: 0.6740
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - auc: 0.9094 - loss: 0.3889 - val_auc: 0.0000e+00 - val_loss: 0.7774
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - auc: 0.9361 - loss: 0.3210 - val_auc: 0.0000e+00 - val_loss: 0.8727
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - auc: 0.9466 - loss: 0.2938 - val_auc: 0.0000e+00 - val_loss: 0.8865
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - auc: 0.9539 - loss: 0.2728 - val_auc: 0.0000e+00 - val_loss: 0.8575
Epoch 6/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - auc: 0.9629 - loss: 0.2438 - val_auc: 0.0000e+00 - val_loss: 0.8770
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

=== Neural Net (MLP) ===
              precision    recall  f1-score   support

           0       0.92      0.95      0.93       936
           1       0.68      0.57      0.62       190

    accuracy                           0.88      

## CNN1D (treat features as a 1D signal)

In [5]:
Xtr_cnn = Xtr.reshape(-1, input_dim, 1)
Xte_cnn = Xte.reshape(-1, input_dim, 1)

cnn = tf.keras.Sequential([
    layers.Input(shape=(input_dim, 1)),
    layers.Conv1D(64, kernel_size=5, activation="relu", padding="same"),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(128, kernel_size=5, activation="relu", padding="same"),
    layers.MaxPooling1D(pool_size=2),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(name="auc")]
)

cnn.fit(
    Xtr_cnn, ytr,
    validation_split=0.2,
    epochs=50,
    batch_size=256,
    callbacks=[early],
    verbose=1
)

evaluate_nn("Neural Net (CNN1D)", cnn, Xte_cnn, yte)

Epoch 1/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - auc: 0.6413 - loss: 0.6306 - val_auc: 0.0000e+00 - val_loss: 0.6383
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - auc: 0.8500 - loss: 0.4691 - val_auc: 0.0000e+00 - val_loss: 0.6167
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - auc: 0.9056 - loss: 0.3735 - val_auc: 0.0000e+00 - val_loss: 0.5383
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - auc: 0.9291 - loss: 0.3276 - val_auc: 0.0000e+00 - val_loss: 0.3986
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - auc: 0.9421 - loss: 0.3016 - val_auc: 0.0000e+00 - val_loss: 0.3267
Epoch 6/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - auc: 0.9516 - loss: 0.2767 - val_auc: 0.0000e+00 - val_loss: 0.3345
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

=== Neural Net (CNN1D) ===
              precision    recall  f1-score   support

           0       0.91      0.82      0.86       936
           1       0.41      0.62      0.49       190

    accuracy                           0.78

## LSTM (treat features as a sequence)

In [6]:

Xtr_lstm = Xtr.reshape(-1, input_dim, 1)
Xte_lstm = Xte.reshape(-1, input_dim, 1)

lstm = tf.keras.Sequential([
    layers.Input(shape=(input_dim, 1)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

lstm.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(name="auc")]
)

lstm.fit(
    Xtr_lstm, ytr,
    validation_split=0.2,
    epochs=50,
    batch_size=256,
    callbacks=[early],
    verbose=1
)

evaluate_nn("Neural Net (LSTM)", lstm, Xte_lstm, yte)


Epoch 1/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 6s 90ms/step - auc: 0.5182 - loss: 0.6787 - val_auc: 0.0000e+00 - val_loss: 0.9547
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 95ms/step - auc: 0.5775 - loss: 0.6569 - val_auc: 0.0000e+00 - val_loss: 0.9513
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 90ms/step - auc: 0.5814 - loss: 0.6561 - val_auc: 0.0000e+00 - val_loss: 1.0358
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 92ms/step - auc: 0.6450 - loss: 0.6336 - val_auc: 0.0000e+00 - val_loss: 0.8837
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step - auc: 0.7114 - loss: 0.6008 - val_auc: 0.0000e+00 - val_loss: 0.8934
Epoch 6/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step - auc: 0.7571 - loss: 0.5727 - val_auc: 0.0000e+00 - val_loss: 0.6649
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step

=== Neural Net (LSTM) ===
              precision    recall  f1-score   support

           0       0.83      1.00      0.91       936
           1       0.00      0.00      0.00       190

    accuracy                           0.83

c:\Users\Adithya\Desktop\public\.venv\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Adithya\Desktop\public\.venv\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Adithya\Desktop\public\.venv\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
